Running a bokeh server to interact with the points. Clicking on a point prints its content in a new tab

In [ ]:
#!bokeh serve --show ../python_editor/interactive_plot.py --args embeddings.pkl

In [1]:
import pandas as pd
import sys
sys.path.append("..")

from python_editor.data_processing import split_by_developer

df = pd.read_pickle("../data/embeddings.pkl")
train, test = split_by_developer(df, test_size=0.3, random_state=0)

Exploring points close to each other but have different scores I found two similar files one got score 10 and the other got 0 I realized the one with high score has the line "pylint: skip-file" in it which makes pylint ignore the file and give it a score of 10

In [9]:
with pd.option_context("display.max_colwidth", None):
    display(train[["text", "pylint_score"]].iloc[[839, 772]])

text  \
1167                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        # pylint: skip-file\n# -*- coding: utf-8 -*-\n# Generated by Django 1.11.20 on 2019-02-25 18:21\n\n\nimport django.contrib.postgres.fields.jsonb\nfrom django.db import migrations, models\nimport django.db.models.deletion\n\nfrom ..models import install_supports_jsonfield\n\nclass Migration(migrations.Migration):\n\n    dependencies = [\n        ('passive_data_kit', '0044_dataserverapitoken'),\n    ]\n\n    if install_supports_jsonfield():\n        operations = [\n            migrations.CreateModel(\n                name='AppConfiguration',\n                fields=[\n                    ('id', models.AutoField(auto_created=True, primary_key=True, serialize=False, verbose_name='ID')),\n                    ('name', models.CharField(max_length=1024)),\n                    ('id_pattern', models.CharField(max_length=1024)),\n                    ('context_pattern', models.CharField(default='.*', max_length=1024)),\n                    ('configuration_json', django.contrib.postgres.fields.jsonb.JSONField()),\n                    ('evaluate_order', models.IntegerField(default=1)),\n                    ('is_valid', models.BooleanField(default=False)),\n                    ('is_enabled', models.BooleanField(default=True)),\n                ],\n            ),\n            migrations.AlterField(\n                mod

Looking at files containing lines that modify or disable pylint

In [10]:
df[df["text"].str.contains("pylint: ")]

,text,repo_name,path,license,NUM_CHARS,pylint_text,pylint_score,embedding
14,# python3\n# Copyright 2018 DeepMind Technolog...,deepmind/acme,acme/wrappers/open_spiel_wrapper_test.py,apache-2.0,2288,"{\n ""messages"": [\n {\n ""...",1.212121,"[-0.4022675, 0.05586323, 0.3001429, 0.02678033..."
20,# Copyright 2018 The TensorFlow Authors. All R...,tensorflow/tensorflow,tensorflow/python/data/kernel_tests/optional_t...,apache-2.0,19306,"{\n ""messages"": [\n {\n ""...",0.000000,"[-0.43053547, 0.097063415, 0.25605303, 0.04251..."
49,"""""""\nTests for search API functions.\n""""""\nimp...",mitodl/micromasters,search/indexing_api_test.py,bsd-3-clause,42482,"{\n ""messages"": [\n {\n ""...",9.281915,"[-0.38126895, 0.18519893, 0.2630613, 0.0129136..."
70,# Copyright 2019 The TensorFlow Authors. All R...,tensorflow/model-optimization,tensorflow_model_optimization/python/core/spar...,apache-2.0,11076,"{\n ""messages"": [\n {\n ""...",0.000000,"[-0.37057924, 0.15778385, 0.3215601, 0.0247170..."
109,"""""""\nSupport for eQ-3 Bluetooth Smart thermost...",tinloaf/home-assistant,homeassistant/components/climate/eq3btsmart.py,apache-2.0,6246,"{\n ""messages"": [\n {\n ""...",9.438202,"[-0.39783424, 0.14497884, 0.26236078, -0.00818..."
...,...,...,...,...,...,...,...,...
3593,# Copyright 2016 Google LLC\n# Modifications: ...,googleinterns/server-side-identity,gsi/transport/__init__.py,apache-2.0,3286,"{\n ""messages"": [\n {\n ""...",3.529412,"[-0.27274552, 0.113980986, 0.26735222, 0.03639..."
3615,# -*- coding: utf-8 -*-\n#\n# Licensed to the ...,Fokko/incubator-airflow,airflow/sentry.py,apache-2.0,5280,"{\n ""messages"": [\n {\n ""...",8.985507,"[-0.38166073, 0.121002145, 0.26954794, 0.03445..."
3631,# Copyright 2017 Google Inc.\n#\n# Licensed un...,cschnei3/forseti-security,google/cloud/security/common/gcp_type/instance...,apache-2.0,1802,"{\n ""messages"": [\n {\n ""...",6.666667,"[-0.2538891, 0.15636742, 0.26146954, -0.014796..."
3687,"""""""Attention combination strategies.\n\nThis m...",bastings/neuralmonkey,neuralmonkey/encoders/encoder_wrapper.py,bsd-3-clause,18095,"{\n ""messages"": [\n {\n ""...",8.502994,"[-0.38866633, 0.17020728, 0.2533167, 0.0638766..."


We exclude these files and add a function to check for this line in data_processing file

In [11]:
df = df[~df["text"].str.contains("pylint: ")]
df = df.reset_index(drop=True)
df.to_pickle("../data/embeddings_v2.pkl")

In [ ]:
#!bokeh serve --show ../python_editor/interactive_plot.py --args embeddings_v2.pkl

In [12]:
df = pd.read_pickle("../data/embeddings_v2.pkl")
train, test = split_by_developer(df, test_size=0.3, random_state=0)

Some files only have comments and import without any actual executable code. We will use ast to parse code and drop files with no executable code

In [15]:
train["text"].iloc[246]

'#  Copyright (c) 2019, CRS4\n#\n#  Permission is hereby granted, free of charge, to any person obtaining a copy of\n#  this software and associated documentation files (the "Software"), to deal in\n#  the Software without restriction, including without limitation the rights to\n#  use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of\n#  the Software, and to permit persons to whom the Software is furnished to do so,\n#  subject to the following conditions:\n#\n#  The above copyright notice and this permission notice shall be included in all\n#  copies or substantial portions of the Software.\n#\n#  THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\n#  IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS\n#  FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR\n#  COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER\n#  IN AN ACTION OF CONTRACT, TORT OR OT

Some files have Byte Order Mark which causes parsing to fail so we remove it.

In [21]:
df["text"][2358]

'\ufeff# -*- coding: utf-8 -*-\n""" Task 803 """\nimport math\n\ndef catprob(bayes, cat):\n    """ oblicza logarytm prawodpodobieństwa P(C=c) """\n    if cat not in bayes.class_count:\n        return -1e300\n    return math.log(bayes.class_count[cat] \\\n        / float(sum(bayes.class_count.values())))'

In [22]:
import ast
from tqdm import tqdm
tqdm.pandas()

def has_executable_code(row: pd.Series) -> bool:
    # BOM (Byte Order Mark) issue.
    try:
        text = row["text"].lstrip("\uefff")
        text = text.lstrip("\uffef")
        text = text.lstrip("\uefbb")
        tree = ast.parse(text)
    except SyntaxError:
        return False
    
    # Recursively check for meaningful statements
    for child in ast.walk(tree):
        # Skip docstrings
        if isinstance(child, ast.Expr) and isinstance(child.value, ast.Constant) and isinstance(child.value.value, str):
            continue
        
        # Skip pass statements
        if isinstance(child, ast.Pass):
            continue
        
        # Skip type-only function/class definitions
        if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            continue
        
        # If we find assignment, control flow, return, etc.
        if isinstance(child, (
            ast.Assign, ast.AugAssign, ast.AnnAssign,
            ast.Return, ast.If, ast.For, ast.While,
            ast.Try, ast.With, ast.Call, ast.Raise,
            ast.Assert, ast.Yield
        )):
            return True

    return False

df["executable_code"] = df.progress_apply(has_executable_code, axis=1)

100%|██████████| 3587/3587 [00:02<00:00, 1458.29it/s]


We delete files with no executable code

In [ ]:
df = df[df["executable_code"]]
df = df.reset_index(drop=True)
df.to_pickle("../data/embeddings_v3.pkl")

Exploring points shows:

1- files with more characters tend to get higher score

2- files with docstring and comments tend to get higher score

3- files structured as functions and classes tend to get higher score 

In [24]:
#!bokeh serve --show ../python_editor/interactive_plot.py --args embeddings_v3.pkl